## RAG with LangChain, Ollama, and Chroma

This notebook uses a baseline LangChain pipeline:
- `JSONLoader` for reading match records
- `RecursiveCharacterTextSplitter` for chunking
- `OllamaEmbeddings` + `Chroma` for retrieval indexing
- `ChatOllama` for answer generation

It keeps JSON handling simple by relying on built-in loaders and splitters.

In [1]:
# Install dependencies (run once)
!pip install -q langchain langchain-community langchain-text-splitters chromadb jq

In [ ]:
# Optional: limit loaded records for faster tests
# LIMIT_MATCHES = 1 -> first 1 records
# LIMIT_MATCHES = None -> all records
LIMIT_MATCHES = 1

# Build jq schema for either full set or a slice
jq_schema = ".[]"
if LIMIT_MATCHES is not None:
    jq_schema = f".[:{LIMIT_MATCHES}]"

LIMIT_MATCHES = 1
Using jq_schema: .[:1]


In [ ]:
from pathlib import Path

project_root = Path(".").resolve()
combined_path = project_root / "cdf_all_matches.json"

Project root: C:\Users\dyury\Desktop\Master Thesis
Combined file exists: True


In [ ]:
from langchain_community.document_loaders import JSONLoader

# Load records using the jq schema configured above
loader = JSONLoader(
    file_path=str(combined_path),
    jq_schema=jq_schema,
    text_content=False,
)

docs_raw = loader.load()

Loaded 1 documents (one per match)
First doc length (chars): 2139244
First 200 chars: [{"match_id": "3895052", "meta": {"competition": {"competition_id": 9, "competition_name": "1. Bundesliga"}, "season_id": 281, "match_id": "3895052", "match_kickoff_time": "2023-08-19T16:30:00.000Z",  ...


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Character-based chunking to keep embedding inputs manageable
splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000,
    chunk_overlap=100,
    length_function=len,
)

documents = splitter.split_documents(docs_raw)

After splitting: 738 chunks


In [ ]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=str(project_root / "chroma_langchain_db"),
    collection_name="bundesliga_langchain",
)

Indexed 738 chunks in Chroma


In [30]:
from langchain_community.chat_models import ChatOllama

llm = ChatOllama(model="llama3.1:8b")
retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

def rag_answer(question: str) -> str:
    # 1) Retrieve relevant chunks from Chroma
    docs = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs)

    # 2) Build prompt from retrieved context
    prompt = f"""
You are a helpful assistant answering questions about Bundesliga football matches.

Use ONLY the information in the context below. If the answer is not in the context,
say that you don't know.

Context:
{context}

Question: {question}

Answer clearly and briefly.
""".strip()

    # 3) Generate answer with local Ollama model
    response = llm.invoke(prompt)
    # Return text content from model response
    return response.content

In [31]:
question = "How many goals were scored during the match 3895052?"
print("QUESTION:", question)
print("\nANSWER:\n")

print(rag_answer(question))

QUESTION: Who is the home team in match 3895052?

ANSWER:

Unfortunately, the provided data does not explicitly mention the home team in match 3895052.
